# Unsupervised Learning

## Learning Objectives
1. Implement Lloyd's algorithm (k-means) from scratch: assign, update, repeat
2. Compare KMeans, DBSCAN, GaussianMixture, and AgglomerativeClustering in sklearn
3. Apply k-means to customer segmentation with the elbow method for K selection
4. Compare PCA, t-SNE, and UMAP for dimensionality reduction and 2D visualization


In [ ]:
import numpy as np
import torch
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — K-Means from Scratch (Lloyd's Algorithm)

In [ ]:
# ── Level 1: Lloyd's algorithm (numpy) ───────────────────────────────────
import numpy as np

np.random.seed(42)

# Synthetic 2D data with 3 clusters
from sklearn.datasets import make_blobs
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)
X = X.astype(np.float32)

def kmeans(X, k, max_iter=100, tol=1e-4):
    # Lloyd's algorithm: assign to nearest centroid, update, repeat.
    n, d = X.shape
    # Initialise centroids as random data points (k-means++ would be better)
    rng      = np.random.default_rng(42)
    idx      = rng.choice(n, k, replace=False)
    centers  = X[idx].copy()
    labels   = np.zeros(n, dtype=int)

    for it in range(max_iter):
        # Assignment step: closest centroid
        dists  = np.linalg.norm(X[:, None] - centers[None, :], axis=2)   # (n, k)
        new_labels = dists.argmin(axis=1)

        # Update step: recompute centroids
        new_centers = np.array([X[new_labels == c].mean(axis=0)
                                 if (new_labels == c).any() else centers[c]
                                 for c in range(k)])
        shift   = np.linalg.norm(new_centers - centers)
        centers = new_centers
        labels  = new_labels
        if it % 10 == 0:
            # Inertia: sum of squared distances to assigned centroid
            inertia = sum(((X[labels == c] - centers[c]) ** 2).sum()
                          for c in range(k))
            print(f"  iter={it:3d}  centroid_shift={shift:.6f}  inertia={inertia:.2f}")
        if shift < tol:
            print(f"  Converged at iteration {it}")
            break
    return labels, centers

print("K-means (k=3):")
labels, centers = kmeans(X, k=3)
purity = max(np.mean(labels == c) for c in range(3))  # simple proxy
print(f"\nCluster sizes: {np.bincount(labels)}")
print(f"Centers:\n{centers.round(2)}")


## Level 2 — Clustering Algorithm Comparison with sklearn

In [ ]:
# ── Level 2: KMeans vs DBSCAN vs GaussianMixture vs Agglomerative ─────────
import numpy as np, time
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

np.random.seed(42)

# Two datasets: blobs (convex) and moons (non-convex)
X_blobs, y_blobs = make_blobs(n_samples=400, centers=3, random_state=42)
X_moons, y_moons = make_moons(n_samples=400, noise=0.07, random_state=42)

scaler = StandardScaler()

configs = {
    "blobs": (scaler.fit_transform(X_blobs), y_blobs),
    "moons": (scaler.fit_transform(X_moons), y_moons),
}

algorithms = {
    "KMeans":       lambda: KMeans(n_clusters=3, random_state=42, n_init=10),
    "GaussianMix":  lambda: GaussianMixture(n_components=3, random_state=42),
    "Agglomerative":lambda: AgglomerativeClustering(n_clusters=3),
    "DBSCAN":       lambda: DBSCAN(eps=0.4, min_samples=5),
}

print(f"{'Dataset':<8}  {'Algorithm':<14}  {'ARI':>6}  {'Time ms':>8}")
print("-" * 45)
for ds_name, (X_sc, y_true) in configs.items():
    for alg_name, alg_fn in algorithms.items():
        alg = alg_fn()
        t0  = time.perf_counter()
        if hasattr(alg, "fit_predict"):
            pred = alg.fit_predict(X_sc)
        else:
            alg.fit(X_sc)
            pred = alg.predict(X_sc)
        tms = (time.perf_counter() - t0) * 1000
        ari  = adjusted_rand_score(y_true, pred)
        print(f"{ds_name:<8}  {alg_name:<14}  {ari:>6.3f}  {tms:>8.1f}")
    print()


## Real-World Example 1 — Customer Segmentation with KMeans

In [ ]:
# ── RW1: customer segmentation — elbow method + cluster profiling ─────────
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# Synthetic e-commerce customer data
n_customers = 500
age          = np.random.normal(35, 12, n_customers).clip(18, 70)
spend_annual = np.random.exponential(1000, n_customers).clip(50, 10000)
visits_month = np.random.poisson(5, n_customers).clip(1, 30)
cart_abandon  = np.random.beta(2, 5, n_customers)   # [0, 1]

X = np.column_stack([age, spend_annual, visits_month, cart_abandon]).astype(np.float32)
feature_names = ["age", "annual_spend", "visits/month", "cart_abandon_rate"]

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# Elbow method: inertia vs k
inertias = []
k_range  = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sc)
    inertias.append(km.inertia_)
    print(f"  k={k}: inertia={km.inertia_:.1f}")

# Find elbow: largest second derivative
diffs  = np.diff(inertias)
diffs2 = np.diff(diffs)
best_k = k_range.start + np.argmax(-diffs2) + 1
print(f"\nElbow at k={best_k}")

# Final clustering with best_k
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels   = km_final.fit_predict(X_sc)

# Profile clusters
print(f"\nCluster profiles (original scale):")
print(f"{'Cluster':<10}", "  ".join(f"{f:<18}" for f in feature_names))
for c in range(best_k):
    mask   = labels == c
    means  = X[mask].mean(axis=0)
    print(f"  {c} (n={mask.sum():<4})",
          "  ".join(f"{v:>18.2f}" for v in means))


## Real-World Example 2 — Anomaly Detection with IsolationForest

In [ ]:
# ── RW2: anomaly detection — IsolationForest ─────────────────────────────
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# Synthetic normal transactions + a few anomalies
n_normal, n_anomaly = 950, 50
X_normal  = np.random.randn(n_normal, 4)                    # typical transactions
X_anomaly = np.random.uniform(-8, 8, (n_anomaly, 4))        # anomalous transactions
X = np.vstack([X_normal, X_anomaly]).astype(np.float32)
y_true = np.array([1] * n_normal + [-1] * n_anomaly)        # 1=normal, -1=anomaly

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# Sweep contamination parameter
print(f"{'Contamination':>14}  {'Precision':>10}  {'Recall':>8}  {'F1':>6}")
print("-" * 45)
for contamination in [0.02, 0.05, 0.08, 0.10, 0.15]:
    iso = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
    pred = iso.fit_predict(X_sc)   # 1 = inlier, -1 = outlier

    tp = ((pred == -1) & (y_true == -1)).sum()
    fp = ((pred == -1) & (y_true ==  1)).sum()
    fn = ((pred ==  1) & (y_true == -1)).sum()
    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else 0
    print(f"{contamination:>14.2f}  {prec:>10.3f}  {recall:>8.3f}  {f1:>6.3f}")

# Best model
best_iso = IsolationForest(contamination=0.05, random_state=42)
scores   = best_iso.fit(X_sc).score_samples(X_sc)   # more negative = more anomalous
print(f"\nAnomaly score stats — normal mean: {scores[:n_normal].mean():.3f}  "
      f"anomaly mean: {scores[n_normal:].mean():.3f}")
print("More negative score = more isolated = more anomalous")


## Real-World Example 3 — PCA vs t-SNE vs UMAP Comparison

In [ ]:
# ── RW3: dimensionality reduction — PCA vs t-SNE vs UMAP ────────────────
import numpy as np, time
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# High-dimensional data (20-dim, 5 clusters)
X_hd, y = make_blobs(n_samples=300, n_features=20, centers=5,
                      cluster_std=2.0, random_state=42)
X_sc = StandardScaler().fit_transform(X_hd)

methods = {}

# PCA (linear)
t0 = time.perf_counter()
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)
methods["PCA"] = (X_pca, time.perf_counter() - t0,
                  pca.explained_variance_ratio_.sum())

# t-SNE (nonlinear, perplexity controls neighborhood size)
t0 = time.perf_counter()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500)
X_tsne = tsne.fit_transform(X_sc)
methods["t-SNE"] = (X_tsne, time.perf_counter() - t0, None)

# UMAP (optional — graceful fallback if not installed)
try:
    import umap
    t0 = time.perf_counter()
    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap  = reducer.fit_transform(X_sc)
    methods["UMAP"] = (X_umap, time.perf_counter() - t0, None)
except ImportError:
    print("UMAP not installed (pip install umap-learn). Skipping.")

print(f"{'Method':<8}  {'Time (s)':>9}  {'Explained Var':>14}  {'Note'}")
print("-" * 60)
for name, (X_2d, elapsed, var) in methods.items():
    var_str = f"{var:.1%}" if var is not None else "N/A (nonlinear)"
    note = "linear, fast, interpretable" if name == "PCA" else            "nonlinear, slow, preserves local structure" if name == "t-SNE" else            "nonlinear, fast, preserves global+local"
    print(f"{name:<8}  {elapsed:>9.2f}  {var_str:>14}  {note}")

# Cluster separation proxy: within-cluster vs between-cluster distance ratio
for name, (X_2d, _, _) in methods.items():
    within = np.mean([np.var(X_2d[y == c], axis=0).sum() for c in np.unique(y)])
    overall_var = np.var(X_2d, axis=0).sum()
    ratio = within / overall_var
    print(f"{name}: within/total variance ratio = {ratio:.3f}  (lower = better separation)")
